# NLP Experiment 2 – Movie review sentiment with Bag of Words (IMDb)
**Runtime:** Python 3 (CPU). Runtime → Run all.

**Bag of Words (BoW):** each review → a vector of word **counts** over the whole vocabulary (word order ignored).
Classifiers: **Logistic Regression** and **Multinomial Naive Bayes**. Labels: 0 = negative, 1 = positive.

In [ ]:
!pip install -q datasets

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

## Load IMDb (25,000 train + 25,000 test reviews, balanced)

In [ ]:
imdb = load_dataset("stanfordnlp/imdb")
train = pd.DataFrame(imdb["train"])
test = pd.DataFrame(imdb["test"])

print("Train:", len(train), " Test:", len(test))
print(train["label"].value_counts())
print(train["text"][0][:300])

## (a) Bag of Words feature extraction

In [ ]:
vectorizer = CountVectorizer(lowercase=True, stop_words="english", max_features=20000)

X_train = vectorizer.fit_transform(train["text"])   # learn vocabulary on TRAIN, then count
X_test = vectorizer.transform(test["text"])         # only count (same vocabulary)
y_train = train["label"]
y_test = test["label"]

print("BoW matrix shape (reviews x words):", X_train.shape)
print("Some vocabulary words:", vectorizer.get_feature_names_out()[1000:1010])

In [ ]:
# what one review looks like as a BoW vector (non-zero counts only)
vocab = vectorizer.get_feature_names_out()
row = X_train[0]
for index, count in zip(row.indices[:15], row.data[:15]):
    print(vocab[index], "->", count)

## (b) Train classifiers

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

nb = MultinomialNB()
nb.fit(X_train, y_train)

print("Training done.")

## (c) Evaluate: accuracy, precision, recall, F1

In [ ]:
def evaluate(name, y_true, y_pred):
    print(name)
    print(f"  Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"  Recall   : {recall_score(y_true, y_pred):.4f}")
    print(f"  F1-score : {f1_score(y_true, y_pred):.4f}")

lr_pred = lr.predict(X_test)
nb_pred = nb.predict(X_test)

evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Naive Bayes", y_test, nb_pred)

In [ ]:
print(classification_report(y_test, lr_pred, target_names=["Negative", "Positive"]))

cm = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix – Logistic Regression")
plt.show()

## Try new reviews

In [ ]:
reviews = [
    "This movie was absolutely fantastic, I loved every minute of it.",
    "Boring, badly written and a complete waste of time.",
    "Amazing acting and an excellent storyline.",
    "I hated it. Terrible and disappointing.",
]
predictions = lr.predict(vectorizer.transform(reviews))

for review, p in zip(reviews, predictions):
    print("Positive" if p == 1 else "Negative", "|", review)